# 🧠 Neural Networks & Deep Learning
**ISLP Chapter 10 · Data Pattern Recognition for the Rest of Us**

> Neural networks learn hierarchical representations from data. For tabular business data, even a shallow network often outperforms linear models. For images, text, and sequences, deep networks are transformative.

### Dataset
**Employee Compensation Analytics**
Predict annual salary from job title, department, years of experience, education, and performance metrics. A real HR/compensation analytics problem where neural networks can capture nonlinear interactions (e.g., seniority × department effects).

### Time: ~65 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
print("\u2713 Setup complete")

In [ ]:
# Employee Compensation Dataset
# Realistic salary data with nonlinear interactions between features
np.random.seed(42)
n = 3000

departments = ['Engineering','Finance','Marketing','Sales','Operations','HR','Legal']
job_levels  = ['Junior','Mid','Senior','Lead','Director','VP']
education   = ['High School','Associate','Bachelor','Master','PhD']

dept = np.random.choice(departments, n, p=[0.30,0.15,0.12,0.18,0.12,0.08,0.05])
level= np.random.choice(job_levels, n, p=[0.20,0.28,0.25,0.15,0.08,0.04])
edu  = np.random.choice(education, n, p=[0.05,0.08,0.45,0.32,0.10])
yoe  = np.random.randint(0, 25, n).astype(float)
perf = np.random.randint(1, 6, n).astype(float)  # 1-5 performance rating
age  = (22 + yoe + np.random.normal(0, 2, n)).clip(22, 65)

# Encode levels
dept_pay  = {'Engineering':1.3,'Finance':1.2,'Legal':1.25,'Marketing':1.0,
             'Sales':0.95,'Operations':0.9,'HR':0.88}
level_mult= {'Junior':0.6,'Mid':0.85,'Senior':1.1,'Lead':1.35,'Director':1.7,'VP':2.2}
edu_bonus = {'High School':0,'Associate':0.03,'Bachelor':0.08,'Master':0.15,'PhD':0.22}

# Nonlinear salary formula: interaction between dept, level, experience
base = 55000
salary = np.array([
    base
    * dept_pay[dept[i]]
    * level_mult[level[i]]
    * (1 + edu_bonus[edu[i]])
    * (1 + 0.025 * min(yoe[i], 15))   # experience premium plateaus
    * (1 + 0.04 * (perf[i] - 3))      # performance premium
    * np.random.lognormal(0, 0.08)     # noise
    for i in range(n)
])

df = pd.DataFrame({
    'Department': dept, 'JobLevel': level, 'Education': edu,
    'YearsExperience': yoe, 'PerformanceRating': perf,
    'Age': age, 'Salary': salary.round(-2)
})

print(f"Employee Compensation dataset: {df.shape}")
print(f"\nSalary range: ${df['Salary'].min():,.0f} — ${df['Salary'].max():,.0f}")
print(f"Median salary: ${df['Salary'].median():,.0f}")
print(f"\nBreakdown by level:")
print(df.groupby('JobLevel')['Salary'].agg(['mean','count']).sort_values('mean').round(0).to_string())

## 📐 Part 1 — From Neuron to Network

**A single neuron computes:**
```
a = f(w₀ + w₁x₁ + w₂x₂ + ... + wₚxₚ)
```
where f is a nonlinear activation function (ReLU: max(0,x)).

**Why layers matter:** Each layer transforms the data into a new representation. Layer 1 might learn "seniority × department" interactions. Layer 2 might learn how those interact with education. The network automatically discovers these combinations from data.

**For tabular data:** Even a shallow network (1-2 hidden layers) often beats linear models by capturing interactions and nonlinearities without explicit feature engineering.

In [ ]:
# Prepare features
le = LabelEncoder()
df_enc = df.copy()
for col in ['Department','JobLevel','Education']:
    df_enc[col] = le.fit_transform(df[col])

X = df_enc.drop('Salary', axis=1).values
y = np.log(df_enc['Salary'].values)  # predict log(salary) — stabilizes variance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

# Compare: Linear Regression vs Neural Network
lr = LinearRegression().fit(X_tr_s, y_tr)
lr_pred = lr.predict(X_te_s)

nn_shallow = MLPRegressor(hidden_layer_sizes=(64,), activation='relu',
                           max_iter=500, random_state=42, early_stopping=True)
nn_shallow.fit(X_tr_s, y_tr)
nn_s_pred = nn_shallow.predict(X_te_s)

nn_deep = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu',
                        max_iter=500, random_state=42, early_stopping=True)
nn_deep.fit(X_tr_s, y_tr)
nn_d_pred = nn_deep.predict(X_te_s)

print(f"{'Model':<30} {'R²':>8} {'MAE ($)':>12} {'RMSE ($)':>12}")
print("-" * 65)
for name, pred in [('Linear Regression', lr_pred),
                   ('Shallow NN (64)', nn_s_pred),
                   ('Deep NN (128-64-32)', nn_d_pred)]:
    r2   = r2_score(y_te, pred)
    mae  = mean_absolute_error(np.exp(y_te), np.exp(pred))
    rmse = np.sqrt(mean_squared_error(np.exp(y_te), np.exp(pred)))
    print(f"  {name:<28} {r2:>8.4f} {mae:>12,.0f} {rmse:>12,.0f}")
print("\nMAE/RMSE are in original $ scale (we exponentiate log predictions)")

In [ ]:
# Visualize: actual vs predicted salary + learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs predicted
y_pred_dollars = np.exp(nn_deep.predict(X_te_s))
y_actual_dollars = np.exp(y_te)

axes[0].scatter(y_actual_dollars/1000, y_pred_dollars/1000,
                alpha=0.3, s=12, color='#1e5fa8')
max_val = max(y_actual_dollars.max(), y_pred_dollars.max()) / 1000
axes[0].plot([0,max_val],[0,max_val],'k--',lw=1.5,alpha=0.5,label='Perfect prediction')
axes[0].set_xlabel('Actual Salary ($000s)')
axes[0].set_ylabel('Predicted Salary ($000s)')
axes[0].set_title(f'Neural Network: Actual vs Predicted Salary\n'
                  f'R² = {r2_score(y_te, nn_deep.predict(X_te_s)):.3f}')
axes[0].legend()

# Residuals by department
pred_all = np.exp(nn_deep.predict(scaler.transform(X)))
residuals_pct = (pred_all - df['Salary'].values) / df['Salary'].values * 100

for dept_name in departments:
    mask = df['Department'] == dept_name
    axes[1].boxplot(residuals_pct[mask], positions=[departments.index(dept_name)],
                    widths=0.6, patch_artist=True,
                    boxprops=dict(facecolor='#1e5fa8', alpha=0.6))

axes[1].axhline(0, color='#e85d20', lw=2, ls='--')
axes[1].set_xticks(range(len(departments)))
axes[1].set_xticklabels([d[:4] for d in departments], rotation=45)
axes[1].set_ylabel('Prediction Error (%)')
axes[1].set_title('Residuals by Department\n(should be centered at 0 for unbiased model)')

plt.tight_layout(); plt.show()

In [ ]:
# Activation functions — why nonlinearity matters
x_range = np.linspace(-4, 4, 200)
activations = {
    'ReLU': np.maximum(0, x_range),
    'Sigmoid': 1/(1+np.exp(-x_range)),
    'Tanh': np.tanh(x_range),
    'Linear (no activation)': x_range,
}
fig, ax = plt.subplots(figsize=(9, 4))
colors_act = ['#1e5fa8','#e85d20','#1a7a45','#888']
for (name, vals), color in zip(activations.items(), colors_act):
    lw = 2.5 if name == 'ReLU' else 1.5
    ls = '-' if name != 'Linear (no activation)' else '--'
    ax.plot(x_range, vals, label=name, color=color, lw=lw, ls=ls)
ax.axhline(0, color='black', lw=0.5, alpha=0.3)
ax.axvline(0, color='black', lw=0.5, alpha=0.3)
ax.set_xlabel('Input value')
ax.set_ylabel('Output value')
ax.set_title('Activation Functions — why nonlinearity is essential\n'
             'Without it, a 10-layer network reduces to a single linear transformation')
ax.legend()
ax.set_ylim(-1.5, 4)
plt.tight_layout(); plt.show()
print("\n\u2714 ReLU is the default for tabular data: fast, avoids vanishing gradients")
print("   Without nonlinear activations, deep networks are mathematically equivalent")
print("   to a single linear regression — no benefit from the extra layers")

In [ ]:
# @title 📝 Quiz — Neural Networks & Deep Learning
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does a single neuron compute? Describe the two steps.
# @markdown **Q2:** Why do neural networks need nonlinear activation functions?
# @markdown **Q3:** What is backpropagation and what problem does it solve?
# @markdown **Q4:** On the compensation dataset, the NN outperformed linear regression. Why?
# @markdown **Q5:** Name one advantage and one disadvantage of neural networks vs Random Forests for tabular HR data.

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Neural Networks & Deep Learning"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username, then click ▶ Run for question-by-question feedback.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re, time
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_quiz_questions():
    """Pull question text from the quiz cell @markdown lines."""
    try:
        import ipynbname
        nb_path = ipynbname.path()
        with open(nb_path) as f:
            nb = json.load(f)
        for cell in nb["cells"]:
            src = "".join(cell.get("source", []))
            if "@title" in src and "Quiz" in src:
                qs = re.findall(r"@markdown \*\*Q\d+:\*\* (.+)", src)
                if qs: return qs
    except Exception:
        pass
    return []

def _call_gemini(prompt, max_retries=3):
    """Call Gemini with retry on 429 rate limit."""
    last_err = None
    for attempt in range(max_retries):
        try:
            import google.generativeai as genai
            import google.auth, google.auth.transport.requests
            creds, _ = google.auth.default()
            creds.refresh(google.auth.transport.requests.Request())
            genai.configure(credentials=creds)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 1500, "temperature": 0.3}
            )
            return result.text, "Gemini via Colab"
        except Exception as e:
            last_err = str(e)
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 2 ** attempt
                print(f"  Rate limit hit — waiting {wait}s before retry {attempt+1}/{max_retries}...")
                time.sleep(wait)
                continue
            break
    # Try stored key as fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini via key"
    except Exception:
        pass
    return None, last_err

def _build_prompt(answers_dict, nb_name, questions):
    answered  = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total   = len(answers_dict)
    n_done    = len(answered)

    # Pair each question with the student answer
    qa_pairs = ""
    for i, (k, v) in enumerate(answers_dict.items(), 1):
        q_text = questions[i-1] if i-1 < len(questions) else f"Question {i}"
        a_text = str(v).strip() if str(v).strip() else "(no answer)"
        qa_pairs += f"Q{i}: {q_text}\nA{i}: {a_text}\n\n"

    return f"""You are a TA grading a student quiz for the "{nb_name}" notebook in a data science course called "Data Pattern Recognition for the Rest of Us" (based on ISLP and business analytics).

{qa_pairs.strip()}

For EACH question:
- Decide if the answer is CORRECT, PARTIALLY CORRECT, or INCORRECT
- A short paraphrase or reasonable approximation counts as CORRECT — do not require exact wording
- For INCORRECT or PARTIAL: name the specific concept they need to review (1 sentence)
- For CORRECT: brief confirmation of what they got right (1 sentence)

Then give an overall summary.

Respond ONLY in this exact JSON format (no markdown fences, no extra text):
{{
  "questions": [
    {{
      "q": 1,
      "status": "<CORRECT|PARTIAL|INCORRECT>",
      "comment": "<one specific sentence>"
    }}
  ],
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent|Good|Needs Review|Incomplete>",
  "summary": "<2 sentences overall: strengths and what to revisit>",
  "review_tip": "<one specific concept, chapter, or notebook to review if they struggled — or 'Great work!' if excellent>"
}}

Scoring guide: CORRECT=1pt, PARTIAL=0.5pt (round to nearest int), INCORRECT=0pt."""

def _show(result, username, nb_name, source, questions):
    STATUS_ICON  = {"CORRECT": "\u2705", "PARTIAL": "\u26a0\ufe0f", "INCORRECT": "\u274c"}
    STATUS_COLOR = {"CORRECT": "\033[92m", "PARTIAL": "\033[93m", "INCORRECT": "\033[91m"}
    R = "\033[0m"
    grade = result.get("grade", "?")
    GRADE_COLOR = {"Excellent":"\033[92m","Good":"\033[94m",
                   "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    GC = GRADE_COLOR.get(grade, "")
    n  = len(answers)
    qs = result.get("quiz_score", 0)
    bar = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*60)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*60)
    print(f"  Student  : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade    : {GC}{grade}{R}")
    print(f"  Score    : {qs}/{n}  [{bar}]")
    print()

    # Question-by-question breakdown
    q_results = result.get("questions", [])
    if q_results:
        print("  \u2500"*29)
        for qr in q_results:
            idx    = qr.get("q", 0) - 1
            status = qr.get("status", "?")
            icon   = STATUS_ICON.get(status, "\u2753")
            color  = STATUS_COLOR.get(status, "")
            comment= qr.get("comment", "")
            q_text = questions[idx] if idx < len(questions) else f"Question {qr.get('q','?')}"
            # Truncate long question text for display
            q_short = q_text[:55] + "..." if len(q_text) > 55 else q_text
            print(f"  {icon} {color}Q{qr.get('q','?')} {status}{R}")
            print(f"     {q_short}")
            if comment:
                for line in textwrap.wrap(comment, 54):
                    print(f"     \u2192 {line}")
            print()

    # Summary
    summary = result.get("summary", "")
    if summary:
        print("  \u2500"*29)
        print("  Overall:")
        for line in textwrap.wrap(summary, 56):
            print(f"  {line}")

    # Review tip
    tip = result.get("review_tip", "")
    if tip and tip != "Great work!":
        print()
        for line in textwrap.wrap(f"\U0001f4d6 Review: {tip}", 56):
            print(f"  {line}")
    elif tip == "Great work!":
        print()
        print("  \U0001f4d6 Great work! Keep going.")

    print("\u2550"*60 + "\n")

def _fallback_grade(answers_dict):
    """Smart fallback — grade on completion only, no length penalty."""
    n  = len(answers_dict)
    nd = sum(1 for v in answers_dict.values() if str(v).strip())
    if nd == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "questions":[],
                "summary":"No answers provided — fill in the quiz above.",
                "review_tip":"Complete the quiz and re-run for AI feedback."}
    elif nd == n:
        return {"quiz_score":n,"grade":"Good",
                "questions":[],
                "summary":f"All {n} questions answered. AI grading was unavailable — re-run to get question-by-question feedback.",
                "review_tip":"Re-run this cell in a few minutes for detailed AI feedback."}
    else:
        return {"quiz_score":nd,"grade":"Needs Review",
                "questions":[],
                "summary":f"{nd}/{n} questions answered. Complete the remaining {n-nd} and re-run.",
                "review_tip":"Answer all questions for full feedback."}

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd       = sum(1 for v in answers.values() if str(v).strip())
    username = GITHUB_USERNAME.strip()
    questions = _get_quiz_questions()

    print(f"\n  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    if username: print(f"  Student  : @{username}")
    print(f"  Grading  : please wait...\n")

    prompt     = _build_prompt(answers, _NB_TITLE, questions)
    raw, src   = _call_gemini(prompt)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            nd2    = sum(1 for v in answers.values() if str(v).strip())
            result = {"quiz_score":nd2,"grade":"Good" if nd2==len(answers) else "Needs Review",
                      "questions":[],"summary":raw[:400],"review_tip":""}
    else:
        if src: print(f"  \u26a0 Gemini unavailable ({src[:60]}) \u2014 showing completion feedback\n")
        result = _fallback_grade(answers)

    _show(result, username, _NB_TITLE, src if raw else None, questions)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*